## **Parte 1: Importación de librerías**

In [11]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [12]:
import pandas as pd
import numpy as np
import pickle

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import StratifiedKFold

print('✅ Librerías importadas correctamente')

✅ Librerías importadas correctamente


### **Parte 2: Carga del dataset**

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import os
print(os.getcwd())

/content


In [18]:
d = pd.read_csv('/content/drive/MyDrive/1_Carrera/Proyecto/Entregable_final/Dataset_comentarios_limpio.csv')
d = d.sample(frac=1)
d = d.reset_index(drop=True)

print(f'Total de comentarios: {len(d)}')
print(f'  Spam    (CLASS=1): {len(d.query("CLASS == 1"))}')
print(f'  No spam (CLASS=0): {len(d.query("CLASS == 0"))}')
print(d.head())

Total de comentarios: 1953
  Spam    (CLASS=1): 1003
  No spam (CLASS=0): 950
                              COMMENT_ID                       AUTHOR  \
0      z13udxpqgtnnxt10o232vhgbspnveld0c             bossdon redhouse   
1    z13tv3oqtkz0exqhg04cjfgqlyrgipmou20                  vuong quang   
2    z13qhxcb2ybzszosx22rh5hwhmmccpjx404  The Silent Troll Defuser HD   
3    z12xxdjrvmynezpqt04chzxjrvqfxntibh0                      lil jay   
4  z131xnwierifxxkj204cgvjxyo3oydb42r40k                YULIOR ZAMORA   

                  DATE                                            CONTENT  \
0                  NaN       I really am madly in love with this woman!!﻿   
1                  NaN                                              Like﻿   
2  2014-11-13 15:47:27  Hey guys can you check my YouTube channel I kn...   
3                  NaN                         Check out my music niggas﻿   
4  2014-09-10 01:35:54  I    loved        it           so       much  ...   

   CLASS  
0      0 

## **Parte 3: Validación cruzada con StratifiedKFold**

In [20]:
kfold = StratifiedKFold(n_splits=5)

# Generamos los índices de cada split
splits = kfold.split(d, d['CLASS'])

# Comprobamos que los splits no se superponen
print('🔀 Índices de test para cada fold:')
for i, (train, test) in enumerate(kfold.split(d, d['CLASS'])):
    print(f'\nSplit {i+1} (primeros 20 índices): {test[:20]}')

🔀 Índices de test para cada fold:

Split 1 (primeros 20 índices): [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]

Split 2 (primeros 20 índices): [381 383 386 387 389 391 393 398 399 400 401 402 403 404 405 406 407 408
 409 410]

Split 3 (primeros 20 índices): [767 770 772 773 774 775 780 785 789 791 792 793 794 795 796 797 798 799
 800 801]

Split 4 (primeros 20 índices): [1172 1173 1174 1176 1177 1178 1179 1180 1181 1182 1183 1184 1185 1186
 1187 1188 1189 1190 1191 1192]

Split 5 (primeros 20 índices): [1555 1558 1560 1561 1562 1563 1567 1569 1571 1572 1573 1574 1575 1576
 1577 1578 1579 1580 1581 1582]


## **Parte 4: Función train_and_test**

In [21]:
def train_and_test(train_idx, test_idx):

    # 1. Extraemos los comentarios
    train_content = d['CONTENT'].iloc[train_idx]
    test_content  = d['CONTENT'].iloc[test_idx]

    # 2. Tokenizador TF-IDF (2000 palabras)
    tokenizer = Tokenizer(num_words=2000)
    tokenizer.fit_on_texts(train_content)

    d_train_inputs = tokenizer.texts_to_matrix(train_content, mode='tfidf')
    d_test_inputs  = tokenizer.texts_to_matrix(test_content,  mode='tfidf')

    # 3. Normalización entre -1 y 1
    d_train_inputs = d_train_inputs / np.amax(np.absolute(d_train_inputs))
    d_test_inputs  = d_test_inputs  / np.amax(np.absolute(d_test_inputs))

    d_train_inputs = d_train_inputs - np.mean(d_train_inputs)
    d_test_inputs  = d_test_inputs  - np.mean(d_test_inputs)

    # 4. Etiquetas one-hot
    d_train_outputs = to_categorical(d['CLASS'].iloc[train_idx])
    d_test_outputs  = to_categorical(d['CLASS'].iloc[test_idx])

    # 5. Red neuronal — MEJORA 2: capa extra de 256 neuronas
    model = Sequential()

    model.add(Dense(512, input_shape=(2000,)))
    model.add(Activation('relu'))
    model.add(Dropout(0.5))

    # ← Nueva capa intermedia
    model.add(Dense(256))
    model.add(Activation('relu'))
    model.add(Dropout(0.3))

    model.add(Dense(2))
    model.add(Activation('softmax'))

    # 6. Compilación
    model.compile(
        loss='categorical_crossentropy',
        optimizer='adamax',
        metrics=['accuracy']
    )

    # 7. MEJORA 1: Early Stopping
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True
    )

    # 8. Entrenamiento con validación y early stopping
    model.fit(
        d_train_inputs,
        d_train_outputs,
        epochs=10,
        batch_size=16,
        validation_split=0.1,   # 10% de train como validación interna
        callbacks=[early_stop],
        verbose=1
    )

    # 9. Evaluación
    scores = model.evaluate(d_test_inputs, d_test_outputs, verbose=0)

    return scores, model, tokenizer   # ← Devolvemos también modelo y tokenizer

## **Parte 5: Ejecución de la validación cruzada**

In [22]:
kfold = StratifiedKFold(n_splits=5)
splits = kfold.split(d, d['CLASS'])

cvscores = []
mejor_accuracy = 0
mejor_modelo = None
mejor_tokenizer = None

for i, (train_idx, test_idx) in enumerate(splits):
    print('\n' + '='*40)
    print('  FOLD ' + str(i+1) + ' / 5')
    print('='*40)

    scores, model, tokenizer = train_and_test(train_idx, test_idx)
    accuracy = scores[1] * 100
    cvscores.append(accuracy)
    print('\n  Precision Fold ' + str(i+1) + ': ' + str(round(accuracy, 2)) + '%')

    # MEJORA 3: guardamos el fold con mejor precisión
    if accuracy > mejor_accuracy:
        mejor_accuracy = accuracy
        mejor_modelo = model
        mejor_tokenizer = tokenizer

# Guardamos el mejor modelo y tokenizer para Streamlit
mejor_modelo.save('modelo_spam.keras')
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(mejor_tokenizer, f)

print('\n✅ Mejor modelo guardado: ' + str(round(mejor_accuracy, 2)) + '%')


  FOLD 1 / 5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.6107 - loss: 0.6404 - val_accuracy: 0.8535 - val_loss: 0.5547
Epoch 2/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8790 - loss: 0.4310 - val_accuracy: 0.9299 - val_loss: 0.2992
Epoch 3/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9310 - loss: 0.2377 - val_accuracy: 0.9363 - val_loss: 0.1947
Epoch 4/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9502 - loss: 0.1592 - val_accuracy: 0.9363 - val_loss: 0.1580
Epoch 5/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9637 - loss: 0.1194 - val_accuracy: 0.9363 - val_loss: 0.1435
Epoch 6/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9665 - loss: 0.1062 - val_accuracy: 0.9427 - val_loss: 0.1409
Epoch 7/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.9722 - loss: 0.0888 - val_accuracy: 0.9618 - val_loss: 0.1285
Epoch 8/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.9772 - loss: 0.0768 - val_accuracy: 0.9618 - v

## **Parte 6: Resultado Final**

In [23]:
print('🎯 Precisión media de la validación cruzada:')
print('%.2f%% (+/- %.2f%%)' % (np.mean(cvscores), np.std(cvscores)))

# Resultado esperado según el libro: ~95.09% (+/- 1.72%)

🎯 Precisión media de la validación cruzada:
94.88% (+/- 0.23%)


In [24]:
# Guardamos el modelo entrenado en formato (.keras)
model.save('modelo_mnist.keras')